# Building disordered supercells: sampling, substitution and displacements

This notebook demonstrates the structure_ops API: sampling a configuration
from an average structure, repeating it into a supercell, substituting
species, checking the composition, and displacing atoms.  It uses the same
COD GaAs CIF fixture as [06_cif_ase_and_diffpy_conversion.ipynb](06_cif_ase_and_diffpy_conversion.ipynb) and the
structure_ops functions from `discuspdf`.

See [docs/STRUCTURE_OPS.md](../docs/STRUCTURE_OPS.md) for the full
documentation.

In [1]:
from pathlib import Path
import os
import sys
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import numpy as np
from discuspdf import Lattice, Structure, repeat, substitute, displace, sample_configuration, check_composition
from discuspdf.cif_io import read_cif

## Read the COD GaAs CIF

In [2]:
# read_cif — read a CIF and expand the asymmetric unit by its space group (needs gemmi, the 'cif' extra)
#   path             CIF file (ASCII)
#   block            data block name; required when the file has several (default None)
#   model            "average" (default; site occupancies) or "configuration" (all occupancies must be 1)
#   allow_loss       False (default) refuses anisotropic ADPs, disorder groups, moments; True warns instead
#   min_separation   overlap check between different sites, Å (default 0.1)
#   returns          CifImport: .structure (unit cell, supercell (1,1,1)), .spacegroup, .loss, …
cif = read_cif("tests/fixtures/interop/gaas_cod_9008845.cif")
s_avg = cif.structure
print(s_avg)
assert s_avg.model == "average"

Structure(n=8, formula=Ga4As4, model='average', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=5.654 Å, b=5.654 Å, c=5.654 Å, alpha=90°, beta=90°, gamma=90°)


## Sample a configuration

In [3]:
# sample_configuration — draw one realised configuration from an average (site-occupancy) structure
#   structure   model="average" Structure; each site's occupancies are the probabilities of its members
#   seed        integer seed (required); a site with total occupancy < 1 may become a vacancy
#   returns     EditResult: .structure (model="configuration", occupancies 1), .ids = removed atom_ids
s_conf = sample_configuration(s_avg, seed=42).structure
print(s_conf)
assert s_conf.model == "configuration"
assert s_conf.n_atoms == 8

Structure(n=8, formula=Ga4As4, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=5.654 Å, b=5.654 Å, c=5.654 Å, alpha=90°, beta=90°, gamma=90°)


## Repeat 4×4×4

In [4]:
# repeat — build a supercell by repeating the structure
#   structure        parent Structure; all pbc must be True
#   reps             (nx, ny, nz), positive integers
#   min_separation   overlap-check threshold in Å (default 0.1; None skips it)
#   returns          EditResult; .structure is the supercell (atom_ids renumbered per cell offset)
s_rep = repeat(s_conf, (4, 4, 4)).structure
print(s_rep)
assert s_rep.n_atoms == 512

Structure(n=512, formula=Ga256As256, model='configuration', supercell=(np.int64(4), np.int64(4), np.int64(4)), a=22.61 Å, b=22.61 Å, c=22.61 Å, alpha=90°, beta=90°, gamma=90°)


## Substitute 10% Ga → In

In [5]:
# substitute — replace some atoms of one species by another (model="configuration" only)
#   old, new     species to replace / new species (non-empty, <= 8 characters, no whitespace)
#   seed         integer seed for the random choice (required)
#   fraction     fraction of the candidates in [0, 1] (rounded to nearest count)  — give exactly one of
#   count        number of atoms to change (>= 0, <= number of candidates)        — fraction / count
#   site         restrict candidates to this site_label (default None = all)
#   new_b_iso    B (Å²) for the substituted atoms; needs a structure with b_iso (default None = unchanged)
#   returns      EditResult; .ids = changed atom_ids, .info["n_changed"]
s_sub = substitute(s_rep, old="Ga", new="In", seed=42, fraction=0.10).structure
n_in = int(np.sum(s_sub.species == "In"))
print(f"In: {n_in}, Ga: {int(np.sum(s_sub.species == 'Ga'))}")
assert n_in == 26

In: 26, Ga: 230


## check_composition

In [6]:
# check_composition — raise ValueError listing every species whose amount differs from `expected`
#   expected     {species: amount}; atom counts for "configuration", occupancy sums for "average"
#   atol, rtol   tolerances on |actual − expected| (defaults 0, 0: exact); missing species count as 0
check_composition(s_sub, {"Ga": 230, "In": 26, "As": 256})
print("OK")

OK


## Displace σ=0.05 Å

In [7]:
# displace — shift selected atoms (model="configuration" only); give exactly one of sigma / vector
#   sigma            Gaussian width per Cartesian component, Å (> 0; needs seed)
#   seed             integer seed for sigma (must be None with vector)
#   vector           fixed shift (3 values) in `coords`: "cartesian" (Å, default) or "fractional"
#   ids / species / site / mask   select the atoms (at most one; default all atoms)
#   min_separation   overlap check after the shift, Å (default 0.1; None skips it)
#   returns          EditResult; .ids = displaced atom_ids
s_disp = displace(s_sub, seed=42, sigma=0.05).structure
print(s_disp)
assert s_disp.n_atoms == 512

Structure(n=512, formula=Ga230As256In26, model='configuration', supercell=(np.int64(4), np.int64(4), np.int64(4)), a=22.61 Å, b=22.61 Å, c=22.61 Å, alpha=90°, beta=90°, gamma=90°)


## Summary

* `sample_configuration` draws a disordered configuration from an average
  structure's occupancies (deterministic with `seed`).
* `repeat` builds a supercell (4×4×4 -> 512 atoms).
* `substitute` replaces a fraction of one species with another; the total
  composition is verified with `check_composition`.
* `displace` adds Gaussian displacements (σ = 0.05 Å) to every atom while
  preserving the atom count.